# nb_ingest_populacao_ibge — Ingestão de Dados Demográficos (Fabric)

**Fonte:** IBGE SIDRA Tabela 6579  
**Escopo:** Santos, Osasco, Mauá e clusters comparativos.

Este notebook realiza a carga da camada Bronze (Raw) e transforma para a camada Silver (Standardized).

In [ ]:
# Importa utilitários core
%run ./nb_utils_ibge

## 1. Bronze Layer — Ingestão

In [ ]:
TABLE_POP = "6579"
VAR_POP   = "9324"

print("=== Iniciando Ingestão de População ===")
df_raw = fetch_sidra_fabric(TABLE_POP, VAR_POP)

if df_raw:
    # Salva na Bronze Raw para histórico
    save_delta(df_raw, "bronze_ibge_populacao_raw")

## 2. Silver Layer — Padronização

Aplicação do schema padrão: `id_municipio | nome_municipio | ano | indicador | valor`

In [ ]:
if df_raw:
    # Padronização usando os códigos universais do SIDRA (D1C, D1N, D3C, V)
    # D1C = município · D2C = código variável (9324) · D3C = ano · V = valor
    df_silver = (
        df_raw
        .select(
            col("D1C").cast("int").alias("id_municipio"),
            trim(regexp_replace(col("D1N"), r"\s*\([A-Z]{2}\)$", "")).alias("nome_municipio"),
            col("D3C").cast("int").alias("ano"),
            lit("populacao_residente").alias("indicador"),
            col("V").cast("double").alias("valor")
        )
        .filter(col("valor").isNotNull())
    )
    
    save_delta(df_silver, "silver_populacao")
    display(df_silver.limit(10))